In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
import sys
import os
import torch.nn as nn
import torch
import torchvision.transforms as T



def conv_layer(channel_in, channel_out, ks=1, stride=1, padding=0, dilation=1, bias=False, bn=True, relu=True, group=1):
    _conv = nn.Conv2d
    sequence = [_conv(channel_in, channel_out, kernel_size=ks, stride=stride, padding=padding, dilation=dilation,
                      bias=bias, groups=group)]
    if bn:
        sequence.append(nn.BatchNorm2d(channel_out))
    if relu:
        sequence.append(nn.ReLU())

    return nn.Sequential(*sequence)


def linear_layer(channel_in, channel_out, bias=False, bn=True, relu=True):
    _linear = nn.Linear
    sequence = [_linear(channel_in, channel_out, bias=bias)]

    if bn:
        sequence.append(nn.BatchNorm1d(channel_out))
    if relu:
        sequence.append(nn.Tanh())

    return nn.Sequential(*sequence)


class mobile_unit(nn.Module):
    dump_patches = True
    def __init__(self, channel_in, channel_out, stride=1, has_half_out=False, num3x3=1):
        super(mobile_unit, self).__init__()
        self.stride = stride
        self.channel_in = channel_in
        self.channel_out = channel_out
        if num3x3 == 1:
            self.conv3x3 = nn.Sequential(
                conv_layer(channel_in, channel_in, ks=3, stride=stride, padding=1, group=channel_in),
            )
        else:
            self.conv3x3 = nn.Sequential(
                conv_layer(channel_in, channel_in, ks=3, stride=1, padding=1, group=channel_in),
                conv_layer(channel_in, channel_in, ks=3, stride=stride, padding=1, group=channel_in),
            )
        self.conv1x1 = conv_layer(channel_in, channel_out)
        self.has_half_out = has_half_out

    def forward(self, x):
        half_out = self.conv3x3(x)
        out = self.conv1x1(half_out)
        if self.stride == 1 and (self.channel_in == self.channel_out):
            out = out + x
        if self.has_half_out:
            return half_out, out
        else:
            return out


def Pool(x, trans, dim=1):
    row, col, value = trans[0].to(x.device), trans[1].to(x.device), trans[2].to(x.device)
    value = value.unsqueeze(-1)
    out = torch.index_select(x, dim, col) * value
    out2 = torch.zeros(x.size(0), row.size(0)//3, x.size(-1)).to(x.device)
    idx = row.unsqueeze(0).unsqueeze(-1).expand_as(out)
    out2 = torch.scatter_add(out2, dim, idx, out)
    return out2


class SpiralDeblock(nn.Module):
    def __init__(self, in_channels, out_channels, indices, meshconv=None):
        """Init a spiral conv block

        Args:
            in_channels (int): input feature dim
            out_channels (int): output feature dim
            indices (tensor): neighbourhood of each hand vertex
            meshconv (optional): conv method, supporting SpiralConv, DSConv. Defaults to SpiralConv.
        """
        super(SpiralDeblock, self).__init__()
        self.conv = meshconv(in_channels, out_channels, indices)
        self.relu = nn.ReLU(inplace=False)
        self.reset_parameters()

    def reset_parameters(self):
        self.conv.reset_parameters()

    def forward(self, x, up_transform):
        out = Pool(x, up_transform)
        out = self.relu(self.conv(out))
        return out

class MLP_res_block(nn.Module):
    def __init__(self, in_dim, hid_dim, dropout=0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(in_dim, eps=1e-6)
        self.fc1 = nn.Linear(in_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, in_dim)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def _ff_block(self, x):
        x = self.fc2(self.dropout1(F.relu(self.fc1(x))))
        return self.dropout2(x)

    def forward(self, x):
        x = x + self._ff_block(self.layer_norm(x))
        return x


class SelfAttn(nn.Module):
    def __init__(self, f_dim, hid_dim=None, n_heads=4, d_q=None, d_v=None, dropout=0.1):
        super().__init__()
        if d_q is None:
            d_q = f_dim // n_heads
        if d_v is None:
            d_v = f_dim // n_heads
        if hid_dim is None:
            hid_dim = f_dim

        self.n_heads = n_heads
        self.d_q = d_q
        self.d_v = d_v
        self.norm = d_q ** 0.5
        self.f_dim = f_dim

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.w_qs = nn.Linear(f_dim, n_heads * d_q)
        self.w_ks = nn.Linear(f_dim, n_heads * d_q)
        self.w_vs = nn.Linear(f_dim, n_heads * d_v)

        self.layer_norm = nn.LayerNorm(f_dim, eps=1e-6)
        self.fc = nn.Linear(n_heads * d_v, f_dim)

        self.ff = MLP_res_block(f_dim, hid_dim, dropout)
    def self_attn(self, x):
        BS, V, f = x.shape

        q = self.w_qs(x).view(BS, -1, self.n_heads, self.d_q).transpose(1, 2)  # BS x h x V x q
        k = self.w_ks(x).view(BS, -1, self.n_heads, self.d_q).transpose(1, 2)  # BS x h x V x q
        v = self.w_vs(x).view(BS, -1, self.n_heads, self.d_v).transpose(1, 2)  # BS x h x V x v

        attn = torch.matmul(q, k.transpose(-1, -2)) / self.norm  # bs, h, V, V
        attn = F.softmax(attn, dim=-1)  # bs, h, V, V
        attn = self.dropout1(attn)

        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(BS, V, -1)
        out = self.dropout2(self.fc(out))
        return out

    def forward(self, x):
        BS, V, f = x.shape
        if f != self.f_dim:
            x = x.permute(0, 2, 1)

        x = x + self.self_attn(self.layer_norm(x))
        x = self.ff(x)

        if f != self.f_dim:
            x = x.permute(0, 2, 1)
        return x




# Advanced modules
class Joint3DDecoder(nn.Module):
    def __init__(self, latent_size, uv_channels, out_channels):
        """Init a 3D decoding with sprial convolution

        Args:
            latent_size (int): feature dim of backbone feature
            out_channels (list): feature dim of each spiral layer
            spiral_indices (list): neighbourhood of each hand vertex
            up_transform (list): upsampling matrix of each hand mesh level
            uv_channel (int): amount of 2D landmark
            meshconv (optional): conv method, supporting SpiralConv, DSConv. Defaults to SpiralConv.
        """
        super(Joint3DDecoder, self).__init__()
        self.latent_size = latent_size
        self.out_channels = out_channels
        self.uv_channels = uv_channels

        self.de_layer_conv = conv_layer(self.latent_size, self.out_channels[- 1], 1,
        bn=False, relu=False)
        self.uv_linear = nn.Linear(3, self.uv_channels)
        self.upsample_1 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_2 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_3 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)

        self.cat_conv1 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_conv2 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_conv3 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_act = nn.SiLU()
        self.cat_conv_final = nn.Conv1d(out_channels[-1] * 3, out_channels[-1], 1, 1)

        self.final_conv = nn.ModuleList([])
        for step in range(len(out_channels) - 1):
            self.final_conv.append(
                nn.ModuleList([
                    nn.Conv1d(out_channels[-1 - step], out_channels[-2 - step], 1, 1),
                    nn.SiLU(),
                    nn.Conv1d(out_channels[-2 - step], out_channels[-2 - step], 1, 1),
                    nn.GroupNorm(8, out_channels[-2 - step]),
                    SelfAttn(out_channels[-2 - step]),
                ])
            )

        self.final_linear1 = nn.Sequential(
            nn.Linear(out_channels[0], 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def index(self, uv, feat):
        uv = uv.unsqueeze(2)  # [B, N, 1, 3]
        samples = torch.nn.functional.grid_sample(feat, uv, align_corners=True)  # [B, C, N, 1]
        return samples[:, :, :, 0]  # [B, C, N]



    def forward(self, uv, x):
        x = self.de_layer_conv(x)
        x_1 = self.index(uv[..., :2], x)
        x_2 = self.index(uv[..., 0::2], x)
        x_3 = self.index(uv[..., 1:], x)

        uv_feat = self.uv_linear(uv).permute(0, 2, 1) # [B, 21, 64]?

        x_1 = torch.bmm(x_1, self.upsample_1.repeat(x.size(0), 1, 1).to(x.device))
        x_2 = torch.bmm(x_2, self.upsample_2.repeat(x.size(0), 1, 1).to(x.device))
        x_3 = torch.bmm(x_3, self.upsample_3.repeat(x.size(0), 1, 1).to(x.device))

        x_1 = torch.cat([x_1, uv_feat], dim=1)
        x_1 = self.cat_conv1(x_1)
        x_2 = torch.cat([x_2, uv_feat], dim=1)
        x_2 = self.cat_conv2(x_2)
        x_3 = torch.cat([x_3, uv_feat], dim=1)
        x_3 = self.cat_conv3(x_3)
        x =  torch.cat([x_1, x_2, x_3], dim=1)

        x = self.cat_conv_final(x)
        x = self.cat_act(x)


        # import pdb; pdb.set_trace()
        for i, (conv, act, conv2, norm, attn) in enumerate(self.final_conv):
            x = conv(x)
            x = act(x)
            x = conv2(x)
            x = norm(x)
            # x = attn(x) + x

        x = x.permute(0, 2, 1)
        x = self.final_linear1(x)
        return x

class Joint3DDecoder_small(nn.Module):
    def __init__(self, latent_size, uv_channels, out_channels):
        """Init a 3D decoding with sprial convolution

        Args:
            latent_size (int): feature dim of backbone feature
            out_channels (list): feature dim of each spiral layer
            spiral_indices (list): neighbourhood of each hand vertex
            up_transform (list): upsampling matrix of each hand mesh level
            uv_channel (int): amount of 2D landmark
            meshconv (optional): conv method, supporting SpiralConv, DSConv. Defaults to SpiralConv.
        """
        super(Joint3DDecoder_small, self).__init__()
        self.latent_size = latent_size
        self.out_channels = out_channels
        self.uv_channels = uv_channels

        self.de_layer_conv = conv_layer(self.latent_size, self.out_channels[- 1], 1,
        bn=False, relu=False)
        self.uv_linear = nn.Linear(3, self.uv_channels)
        self.upsample_1 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_2 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)
        self.upsample_3 = nn.Parameter(torch.ones([21, 21])*0.01, requires_grad=True)

        self.cat_conv1 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_conv2 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_conv3 = nn.Conv1d(out_channels[-1] + uv_channels, out_channels[-1], 3, 1, 1)
        self.cat_act = nn.SiLU()
        self.cat_conv_final = nn.Conv1d(out_channels[-1] * 3, out_channels[-1], 1, 1)

        self.final_conv = nn.ModuleList([])
        for step in range(len(out_channels) - 1):
            self.final_conv.append(
                nn.ModuleList([
                    nn.Conv1d(out_channels[-1 - step], out_channels[-2 - step], 1, 1),
                    nn.SiLU(),
                    nn.Conv1d(out_channels[-2 - step], out_channels[-2 - step], 1, 1),
                    nn.GroupNorm(8, out_channels[-2 - step]),
                    SelfAttn(out_channels[-2 - step]),
                ])
            )

        self.final_linear1 = nn.Sequential(
            nn.Linear(out_channels[0], 64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )

    def index(self, uv, feat):
        uv = uv.unsqueeze(2)  # [B, N, 1, 3]
        samples = torch.nn.functional.grid_sample(feat, uv, align_corners=True)  # [B, C, N, 1]
        return samples[:, :, :, 0]  # [B, C, N]



    def forward(self, uv, x):
        x = self.de_layer_conv(x)
        x_1 = self.index(uv[..., :2], x)
        x_2 = self.index(uv[..., 0::2], x)
        x_3 = self.index(uv[..., 1:], x)

        uv_feat = self.uv_linear(uv).permute(0, 2, 1) # [B, 21, 64]?

        x_1 = torch.bmm(x_1, self.upsample_1.repeat(x.size(0), 1, 1).to(x.device))
        x_2 = torch.bmm(x_2, self.upsample_2.repeat(x.size(0), 1, 1).to(x.device))
        x_3 = torch.bmm(x_3, self.upsample_3.repeat(x.size(0), 1, 1).to(x.device))

        x_1 = torch.cat([x_1, uv_feat], dim=1)
        x_1 = self.cat_conv1(x_1)
        x_2 = torch.cat([x_2, uv_feat], dim=1)
        x_2 = self.cat_conv2(x_2)
        x_3 = torch.cat([x_3, uv_feat], dim=1)
        x_3 = self.cat_conv3(x_3)
        x =  torch.cat([x_1, x_2, x_3], dim=1)

        x = self.cat_conv_final(x)
        x = self.cat_act(x)


        # import pdb; pdb.set_trace()
        for i, (conv, act, conv2, norm, attn) in enumerate(self.final_conv):
            x = conv(x)
            x = act(x)
            # x = conv2(x)
            x = norm(x)
            # x = attn(x) + x

        x = x.permute(0, 2, 1)
        x = self.final_linear1(x)
        return x

class DenseBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//4)
        self.conv2 = mobile_unit(channel_in*5//4, channel_in//4)
        self.conv3 = mobile_unit(channel_in*6//4, channel_in//4)
        self.conv4 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        out4 = self.conv4(comb3)
        comb4 = torch.cat((comb3, out4),dim=1)
        return comb4


class DenseBlock2(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in//2)
        self.conv2 = mobile_unit(channel_in*3//2, channel_in//2)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        return comb2


class DenseBlock3(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock3, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in)
        self.conv2 = mobile_unit(channel_in*2, channel_in)
        self.conv3 = mobile_unit(channel_in*3, channel_in)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((comb1, out2),dim=1)
        out3 = self.conv3(comb2)
        comb3 = torch.cat((comb2, out3),dim=1)
        return comb3


class DenseBlock2_noExpand(nn.Module):
    dump_patches = True

    def __init__(self, channel_in):
        super(DenseBlock2_noExpand, self).__init__()
        self.channel_in = channel_in
        self.conv1 = mobile_unit(channel_in, channel_in*3//4)
        self.conv2 = mobile_unit(channel_in*7//4, channel_in//4)

    def forward(self, x):
        out1 = self.conv1(x)
        comb1 = torch.cat((x, out1),dim=1)
        out2 = self.conv2(comb1)
        comb2 = torch.cat((out1, out2),dim=1)
        return comb2


class SenetBlock(nn.Module):
    dump_patches = True

    def __init__(self, channel, size):
        super(SenetBlock, self).__init__()
        self.size = size
        self.globalAvgPool = nn.AdaptiveAvgPool2d((1, 1))
        self.channel = channel
        self.fc1 = linear_layer(self.channel, min(self.channel//2, 256))
        self.fc2 = linear_layer(min(self.channel//2, 256), self.channel, relu=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        original_out = x
        pool = self.globalAvgPool(x)
        pool = pool.view(pool.size(0), -1)
        fc1 = self.fc1(pool)
        out = self.fc2(fc1)
        out = self.sigmoid(out)
        out = out.view(out.size(0), out.size(1), 1, 1)

        return out * original_out


class DenseStack(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel):
        super(DenseStack, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2, 32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4,16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4, 4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.dense3(d2))
        u1 = self.upsample1(self.senet4(self.thrink1(d3)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.upsample3(self.senet6(self.thrink3(us2)))
        return u3




class DenseStack2(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel, final_upsample=True, ret_mid=False):
        super(DenseStack2, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2,32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4, 16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense3 = DenseBlock(input_channel*4)
        self.senet3 = SenetBlock(input_channel*8,8)
        self.transition3 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*8)
        self.dense5 = DenseBlock2_noExpand(input_channel*8)
        self.thrink1 = nn.Sequential(mobile_unit(input_channel*8, input_channel*4, num3x3=1), mobile_unit(input_channel*4, input_channel*4, num3x3=2))
        self.senet4 = SenetBlock(input_channel*4,4)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)
        self.final_upsample = final_upsample
        if self.final_upsample:
            self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.ret_mid = ret_mid

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.transition3(self.senet3(self.dense3(d2)))
        d4 = self.dense5(self.dense4(d3))
        u1 = self.upsample1(self.senet4(self.thrink1(d4)))
        us1 = d2 + u1
        u2 = self.upsample2(self.senet5(self.thrink2(us1)))
        us2 = d1 + u2
        u3 = self.senet6(self.thrink3(us2))
        if self.final_upsample:
            u3 = self.upsample3(u3)
        if self.ret_mid:
            return u3, u2, u1, d4
        else:
            return u3, d4



class DenseStack3(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel):
        super(DenseStack3, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2, 32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4,16)
        self.transition2 = nn.AvgPool2d(2)
        # self.dense4 = DenseBlock2_noExpand(input_channel*4)
        # self.dense5 = DenseBlock2_noExpand(input_channel*4)
        self.thrink2 = nn.Sequential(
            mobile_unit(input_channel*4, input_channel*2, num3x3=1),
            mobile_unit(input_channel*2, input_channel*2, num3x3=2)
        )

        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(
            mobile_unit(input_channel*2, input_channel*2, num3x3=1),
            mobile_unit(input_channel*2, output_channel, num3x3=2)
        )
        self.senet6 = SenetBlock(output_channel,16)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        u2 = self.upsample2(self.senet5(self.thrink2(d2)))
        us2 = d1 + u2
        u3 = self.upsample3(self.senet6(self.thrink3(us2)))
        return u3

class DenseStack4(nn.Module):
    dump_patches = True

    def __init__(self, input_channel, output_channel, final_upsample=True, ret_mid=False):
        super(DenseStack4, self).__init__()
        self.dense1 = DenseBlock2(input_channel)
        self.senet1 = SenetBlock(input_channel*2,32)
        self.transition1 = nn.AvgPool2d(2)
        self.dense2 = DenseBlock(input_channel*2)
        self.senet2 = SenetBlock(input_channel*4, 16)
        self.transition2 = nn.AvgPool2d(2)
        self.dense4 = DenseBlock2_noExpand(input_channel*4)
        # self.dense5 = DenseBlock2_noExpand(input_channel*4)
        self.thrink2 = nn.Sequential(mobile_unit(input_channel*4, input_channel*2, num3x3=1), mobile_unit(input_channel*2, input_channel*2, num3x3=2))
        self.senet5 = SenetBlock(input_channel*2,8)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.thrink3 = nn.Sequential(mobile_unit(input_channel*2, input_channel*2, num3x3=1), mobile_unit(input_channel*2, output_channel, num3x3=2))
        self.senet6 = SenetBlock(output_channel,16)



    def forward(self, x):
        d1 = self.transition1(self.senet1(self.dense1(x)))
        d2 = self.transition2(self.senet2(self.dense2(d1)))
        d3 = self.dense4(d2)
        u2 = self.upsample2(self.senet5(self.thrink2(d3)))
        us2 = d1 + u2
        return self.senet6(self.thrink3(us2))



class DenseStack_Backnone(nn.Module):
    def __init__(self, input_channel=64, out_channel=21, latent_size=256, kpts_num=21):
        super(DenseStack_Backnone, self).__init__()
        self.pre_layer = nn.Sequential(
            conv_layer(3, input_channel // 4, 3, 2, 1),
            mobile_unit(input_channel // 4, input_channel // 4)
        )
        self.thrink = conv_layer(input_channel , input_channel)
        self.dense_stack1 = DenseStack3(input_channel, out_channel)
        self.stack1_remap = conv_layer(out_channel, out_channel)

        self.thrink2 = conv_layer((out_channel + input_channel), input_channel)
        self.dense_stack2 = DenseStack4(input_channel, out_channel, final_upsample=False)

        # self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.reduce = conv_layer(out_channel, kpts_num, 1, bn=False, relu=False)
        self.uv_reg = nn.Sequential(
            linear_layer(latent_size, 32, bn=False),
            linear_layer(32, 3, bn=False, relu=False)
        )
        self.reorg = conv_layer(input_channel // 4, input_channel, 3, 2, 1) # Reorg()


    def forward(self, x):
        pre_out = self.pre_layer(x)
        # import pdb; pdb.set_trace()
        pre_out_reorg = self.reorg(pre_out)
        thrink = self.thrink(pre_out_reorg)
        stack1_out = self.dense_stack1(thrink)
        stack1_out_remap = self.stack1_remap(stack1_out)
        input2 = torch.cat((stack1_out_remap, pre_out_reorg),dim=1)
        thrink2 = self.thrink2(input2)
        stack2_out = self.dense_stack2(thrink2)
        # import pdb; pdb.set_trace()
        uv_reg = self.uv_reg(self.reduce(stack2_out).view(stack2_out.shape[0], 21, -1))
        # uv_reg = self.uv_reg(stack2_out.view(stack2_out.shape[0], 21, -1))

        return uv_reg





class MobRecon_DS(nn.Module):
    def __init__(self, ):
        super(MobRecon_DS, self).__init__()
        self.backbone = DenseStack_Backnone(latent_size=784)

    def forward(self, x):
        pred_joint = self.backbone(x)

        return pred_joint


In [ ]:
model = MobRecon_DS()
#pth_path = '/content/drive/MyDrive/KCVL/etri/ETRI_224.pth'
pth_path = '/content/drive/MyDrive/KCVL/etri/Small_model_input224_25D_color_augmentation.pth'
model.load_state_dict(torch.load(pth_path), strict=False)
model.eval()

In [ ]:
!pip install onnx onnxruntime

In [ ]:
input_image = torch.ones((1,3,224,224))
b = model(input_image)
b.shape

In [ ]:
input_image = torch.zeros((1,3,224,224))
#torch.onnx.export(model, input_image, "/content/drive/MyDrive/KCVL/etri/mob_ckpt.onnx",opset_version=11, input_names=['input0'], output_names=['output0'])
torch.onnx._export(model, input_image, "/content/drive/MyDrive/KCVL/etri/224_color_aug.onnx", export_params=True, verbose=False, input_names=['input0'], output_names=['output0'])

In [ ]:
import onnx
from onnx import shape_inference
path = "/content/drive/MyDrive/KCVL/etri/mob_ckpt.onnx"
save_path = "/content/drive/MyDrive/KCVL/etri/mob_ckpt_shape.onnx"
onnx.save(onnx.shape_inference.infer_shapes(onnx.load(path)), save_path)